# 00 — Environment tests and GPU pilot

**Purpose:** Check free GPU access, local safety tests, packages, tokenizer alignment, and optional 2-step LoRA/full timing pilots.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# 0. CLONE + ENVIRONMENT SETUP — RUN FIRST (Kaggle/Colab).
# Internet must be ON. The FIRST run may install libraries and STOP for a kernel restart.
# After restarting the kernel, rerun this cell; it will skip installation if versions match.
from pathlib import Path
import os, sys, subprocess, importlib.metadata as metadata

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
PROJECT_FOLDER = "Urdu_Qwen_Research_Notebooks"
PULL_LATEST = False  # Set True after pushing changes to GitHub; requires a clean Git checkout.

if Path('/kaggle/working').is_dir():
    WORKSPACE = Path('/kaggle/working')
elif Path('/content').is_dir():
    WORKSPACE = Path('/content')
else:
    WORKSPACE = Path.cwd()
REPO_DIR = WORKSPACE / 'nlp_research_proj'
ROOT = REPO_DIR / PROJECT_FOLDER

if (REPO_DIR / '.git').is_dir():
    if PULL_LATEST:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a Git clone. Rename/remove it after backing up outputs.')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
if not (ROOT / 'requirements.txt').is_file() or not (ROOT / 'project_utils.py').is_file():
    raise FileNotFoundError(f'Expected complete project at {ROOT}. Check the GitHub folder layout.')

# This environment was selected for a stable Transformers 4 training API.
if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError(f'Python {sys.version.split()[0]} not in tested support range 3.10–3.12.')
REQUIRED = {
    'transformers': '4.51.3',
    'peft': '0.15.2',
    'accelerate': '1.6.0',
    'datasets': '3.5.0',
    'tokenizers': '0.21.1',
    'huggingface-hub': '0.30.2',
    'safetensors': '0.5.3',
    'scikit-learn': '1.6.1',
    'pandas': '2.2.3',
    'numpy': '1.26.4',
    'matplotlib': '3.10.1',
    'psutil': '7.0.0',
    'ipywidgets': '8.1.6',
    'nbformat': '5.10.4',
    'pytest': '8.3.5',
}

def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

mismatch = {name: (installed_version(name), version)
            for name, version in REQUIRED.items()
            if installed_version(name) != version}
torchao = installed_version('torchao')
if mismatch or torchao is not None:
    print('Installing the project-locked environment; CUDA-enabled PyTorch is left untouched.')
    if mismatch:
        print('Packages to align:', mismatch)
    if torchao is not None:
        print('Removing optional incompatible torchao:', torchao)
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt'),
                    '--upgrade-strategy', 'only-if-needed'], check=True)
    raise RuntimeError('Installation finished. RESTART THE KAGGLE/COLAB KERNEL, then rerun this FIRST cell. '
                       'Do not train before restarting.')

# No package installation needed: import the shared research code only now.
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()

# Catch future API drift before downloading a 0.5B model.
import inspect
from transformers import TrainingArguments, Trainer
sig = inspect.signature(TrainingArguments).parameters
needed = ['output_dir', 'group_by_length', 'warmup_ratio', 'eval_strategy']
missing = [name for name in needed if name not in sig]
if missing:
    raise RuntimeError(f'Transformers does not match our 4.51.3 training API. Missing: {missing}')
api = check_training_api()
print('Ready. Project:', ROOT)
print('Transformers:', api['transformers'], '| PEFT:', api['peft'])
print('Trainer token argument:', api['trainer_tokenizer_key'])
print('No installation needed; proceed to the next cell.')


## A. Environment and safety checks

Run this notebook **first**. The first three cells work without downloading Qwen. If packages are missing, run the installation cell in Kaggle/Colab (restart the kernel once after installation). This notebook does not generate paper results.

In [ ]:
import sys, platform, importlib.util
print('Python:', sys.version.split()[0], 'OS:', platform.platform())
for pkg in ['torch','transformers','peft','accelerate','datasets','sklearn','numpy','pandas']:
    spec = importlib.util.find_spec(pkg)
    print(f'{pkg:15}', 'installed' if spec else 'MISSING')
try:
    import torch
    print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0)); print('VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
except ImportError:
    print('Install PyTorch in Kaggle/Colab using the platform-default build.')

### Environment installation is automatic

The first code cell clones your repository, checks all pinned versions, installs only if necessary, and stops for a kernel restart. Do not manually run another `pip install -U` cell.


In [ ]:
# Installation is handled by the first code cell.
print('Environment setup is handled by Cell 1. Do not change package versions mid-run.')


In [ ]:
# Offline unit tests: no model download or GPU required.
import subprocess
command = [sys.executable, '-m', 'pytest', '-q', str(ROOT/'tests')]
res = subprocess.run(command, cwd=ROOT, capture_output=True, text=True)
print(res.stdout[-4000:], res.stderr[-1000:]); assert res.returncode == 0, 'Unit tests failed'

## B. Optional tokenizer and model smoke test

Set `RUN_MODEL_SMOKE=True` only once packages are installed and internet is available. This loads the real 0.5B model and checks that exactly the answer tokens receive loss.

In [ ]:
RUN_MODEL_SMOKE = False
if RUN_MODEL_SMOKE:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(cfg()['model_id'])
    labels = DEFAULT_LABELS
    example = encode_supervised(tokenizer, 'پاکستان نے کرکٹ میچ جیت لیا', 'sports', labels, 'ur', max_length=256)
    assert len(example['input_ids']) == len(example['labels'])
    assert any(x != -100 for x in example['labels'])
    print('Prompt tokens ignored:', sum(x == -100 for x in example['labels']))
    print('Supervised tokens:',sum(x != -100 for x in example['labels']))
    model, tok = load_model(cfg()['model_id'])
    import torch
    with torch.no_grad():
        batch = supervised_collator(tok)([example]); batch = {k:v.to(next(model.parameters()).device) for k,v in batch.items()}
        loss = model(**batch).loss
    print('Real model loss:',float(loss)); assert torch.isfinite(loss).item()
    del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None

## C. Optional two-step GPU training pilot

Run `01_preprocess_data` first. **This is not a scientific result**: it measures real training speed and checks whether full fine-tuning fits. Test LoRA first, then full. Stop and reduce `config.json → max_length` to 192 or 128 if full training runs out of GPU memory. Use two training steps, not an entire epoch. Both pilots use the **same Adafactor optimizer**, so the optimizer is not a hidden difference.

In [ ]:
RUN_GPU_PILOT = False  # Set True after 01_preprocess_data and GPU enabled
if RUN_GPU_PILOT:
    import torch
    assert torch.cuda.is_available(), 'GPU required for timing pilot'
    for method in ['lora','full']:
        try:
            r = train_run(method, 'sib200', size=5, seed=42, rank=8, max_steps=2, pilot=True,
                          run_name=f'pilot_{method}_{int(__import__("time").time())}')
            print(method, '2 steps:', r['wall_seconds'], 'seconds;', 'VRAM GiB:', r['peak_gpu_allocated_gib'])
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f'{method}: OOM. Reduce sequence length / keep batch=1, or try another available GPU.')

## D. Runtime estimation after pilot

A rough estimate from a two-step run can be misleading because it includes model loading, initialization, and warmup. Prefer a 10–20-step pilot for better numbers. No result should be reported as a measured full-training time until that training actually completes.

In [ ]:
pilots = [r for r in read_jsonl(ROOT/'outputs/metrics/train_runs.jsonl') if r.get('pilot')]
for r in pilots:
    print(r['run_id'], 'steps=',r['global_steps'], 'total wall min=',round(r['wall_seconds']/60,1),
          'avg sec/step incl. setup=',round(r['wall_seconds']/max(1,r['global_steps']),2))